# Config Cluster

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, split, lit

spark = SparkSession.builder.appName("hw-3-1").config("spark.sql.autoBroadcastJoinThreshold", "-1").getOrCreate()

25/08/08 10:05:23 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [4]:
print("hello world")

hello world


# matchesBucketed

In [3]:
matchesBucketed = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/matches.csv")

matchesBucketed.printSchema()

[Stage 1:=============================>                             (1 + 1) / 2]

root
 |-- match_id: string (nullable = true)
 |-- mapid: string (nullable = true)
 |-- is_team_game: boolean (nullable = true)
 |-- playlist_id: string (nullable = true)
 |-- game_variant_id: string (nullable = true)
 |-- is_match_over: boolean (nullable = true)
 |-- completion_date: timestamp (nullable = true)
 |-- match_duration: string (nullable = true)
 |-- game_mode: string (nullable = true)
 |-- map_variant_id: string (nullable = true)



In [5]:
spark.sql("""DROP TABLE IF EXISTS bootcamp.matches_bucketed3""")
bucketedDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed3 (
 match_id STRING,
 mapid STRING,
 is_team_game BOOLEAN,
 playlist_id STRING,
 completion_date TIMESTAMP
)
USING iceberg
PARTITIONED BY (completion_date, bucket(16, match_id));
"""
spark.sql(bucketedDDL)

DataFrame[]

In [6]:
(
matchesBucketed.select("match_id", "mapid", "is_team_game", "playlist_id", "completion_date")
.write.mode("append")
.partitionBy("completion_date")
.bucketBy(16, "match_id").saveAsTable("bootcamp.matches_bucketed3")
)

In [7]:
spark.table("bootcamp.matches_bucketed3").printSchema()

root
 |-- match_id: string (nullable = true)
 |-- mapid: string (nullable = true)
 |-- is_team_game: boolean (nullable = true)
 |-- playlist_id: string (nullable = true)
 |-- completion_date: timestamp (nullable = true)



In [5]:
spark.table("bootcamp.matches_bucketed3").count()

24025

In [29]:
spark.table("bootcamp.matches_bucketed3").show(1)

+--------------------+--------------------+------------+--------------------+-------------------+
|            match_id|               mapid|is_team_game|         playlist_id|    completion_date|
+--------------------+--------------------+------------+--------------------+-------------------+
|fa16e9f6-3aeb-47d...|cdb934b0-f206-11e...|        true|892189e9-d712-4bd...|2016-01-13 00:00:00|
+--------------------+--------------------+------------+--------------------+-------------------+
only showing top 1 row



# matchDetailsBucketed

In [9]:
matchDetailsBucketed = ( spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/match_details.csv")
                       )

matchDetailsBucketed.printSchema()

[Stage 9:==============>                                            (1 + 3) / 4]

root
 |-- match_id: string (nullable = true)
 |-- player_gamertag: string (nullable = true)
 |-- previous_spartan_rank: integer (nullable = true)
 |-- spartan_rank: integer (nullable = true)
 |-- previous_total_xp: integer (nullable = true)
 |-- total_xp: integer (nullable = true)
 |-- previous_csr_tier: integer (nullable = true)
 |-- previous_csr_designation: integer (nullable = true)
 |-- previous_csr: integer (nullable = true)
 |-- previous_csr_percent_to_next_tier: integer (nullable = true)
 |-- previous_csr_rank: integer (nullable = true)
 |-- current_csr_tier: integer (nullable = true)
 |-- current_csr_designation: integer (nullable = true)
 |-- current_csr: integer (nullable = true)
 |-- current_csr_percent_to_next_tier: integer (nullable = true)
 |-- current_csr_rank: integer (nullable = true)
 |-- player_rank_on_team: integer (nullable = true)
 |-- player_finished: boolean (nullable = true)
 |-- player_average_life: string (nullable = true)
 |-- player_total_kills: integer (nu

In [10]:
spark.sql("""DROP TABLE IF EXISTS bootcamp.match_details_bucketed3""")
bucketedDetailsDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed3 (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketedDetailsDDL)

DataFrame[]

In [11]:
(
matchDetailsBucketed.select("match_id", "player_gamertag", "player_total_kills", "player_total_deaths")
.write.mode("append")
.bucketBy(16, "match_id").saveAsTable("bootcamp.match_details_bucketed3")
)

In [6]:
spark.table("bootcamp.match_details_bucketed3").count()

151761

In [30]:
spark.table("bootcamp.match_details_bucketed3").show(1)

+--------------------+---------------+------------------+-------------------+
|            match_id|player_gamertag|player_total_kills|player_total_deaths|
+--------------------+---------------+------------------+-------------------+
|f8852913-2ccf-46f...|    OneWingKing|                 7|                  6|
+--------------------+---------------+------------------+-------------------+
only showing top 1 row



# join difference

In [13]:
df = matchesBucketed.join(matchDetailsBucketed, on = ["match_id"], how = "inner")
df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#17, mapid#18, is_team_game#19, playlist_id#20, game_variant_id#21, is_match_over#22, completion_date#23, match_duration#24, game_mode#25, map_variant_id#26, player_gamertag#93, previous_spartan_rank#94, spartan_rank#95, previous_total_xp#96, total_xp#97, previous_csr_tier#98, previous_csr_designation#99, previous_csr#100, previous_csr_percent_to_next_tier#101, previous_csr_rank#102, current_csr_tier#103, current_csr_designation#104, current_csr#105, current_csr_percent_to_next_tier#106, ... 21 more fields]
   +- BroadcastHashJoin [match_id#17], [match_id#92], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=173]
      :  +- Filter isnotnull(match_id#17)
      :     +- FileScan csv [match_id#17,mapid#18,is_team_game#19,playlist_id#20,game_variant_id#21,is_match_over#22,completion_date#23,match_duration#24,game_mode#25,map_variant_id#26

25/07/26 02:18:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [14]:
df2 = spark.table("bootcamp.matches_bucketed").join(spark.table("bootcamp.match_details_bucketed"), on = ["match_id"], how = "inner")
df2.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#247, is_team_game#248, playlist_id#249, completion_date#250, player_gamertag#256, player_total_kills#257, player_total_deaths#258]
   +- BroadcastHashJoin [match_id#247], [match_id#255], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=192]
      :  +- BatchScan demo.bootcamp.matches_bucketed[match_id#247, is_team_game#248, playlist_id#249, completion_date#250] demo.bootcamp.matches_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []
      +- BatchScan demo.bootcamp.match_details_bucketed[match_id#255, player_gamertag#256, player_total_kills#257, player_total_deaths#258] demo.bootcamp.match_details_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []




# Moving forward - medal_matches_players

In [12]:
medal_matches_players_bucketed = ( spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/medals_matches_players.csv")
                        .withColumnRenamed('player_gamertag', 'player_gamertag_mmp')
                       )

medal_matches_players_bucketed.printSchema()

[Stage 20:>                                                         (0 + 4) / 4]

root
 |-- match_id: string (nullable = true)
 |-- player_gamertag_mmp: string (nullable = true)
 |-- medal_id: long (nullable = true)
 |-- count: integer (nullable = true)



In [13]:
spark.sql("""DROP TABLE IF EXISTS bootcamp.medal_matches_players_bucketed3""")
bucketedDetailsDDL = """
CREATE OR REPLACE TABLE bootcamp.medal_matches_players_bucketed3 (
    match_id STRING,
    player_gamertag_mmp STRING,
    medal_id BIGINT,
    count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketedDetailsDDL)

DataFrame[]

In [14]:
(
medal_matches_players_bucketed.select("match_id", "player_gamertag_mmp", "medal_id", "count")
.write.mode("append")
.bucketBy(16, "match_id").saveAsTable("bootcamp.medal_matches_players_bucketed3")
)

In [15]:
spark.table("bootcamp.medal_matches_players_bucketed3").count()

755229

In [31]:
spark.table("bootcamp.medal_matches_players_bucketed3").show(1)

+--------------------+-------------------+---------+-----+
|            match_id|player_gamertag_mmp| medal_id|count|
+--------------------+-------------------+---------+-----+
|e4bff834-89d3-482...|           EcZachly|298813630|    2|
+--------------------+-------------------+---------+-----+
only showing top 1 row



# Small tables - medals and maps

In [8]:
medals = ( spark.read.option("header", "true")
          .option("inferSchema", "true")
          .csv("/home/iceberg/data/medals.csv")
          )

maps = ( spark.read.option("header", "true")
          .option("inferSchema", "true")
          .csv("/home/iceberg/data/maps.csv")
          .withColumnRenamed('name', 'map_name')
          .withColumnRenamed('description', 'map_description')
          )

print(medals.count(), medals.columns, maps.count(), maps.columns, sep = '\n')

183
['medal_id', 'sprite_uri', 'sprite_left', 'sprite_top', 'sprite_sheet_width', 'sprite_sheet_height', 'sprite_width', 'sprite_height', 'classification', 'description', 'name', 'difficulty']
40
['mapid', 'map_name', 'map_description']


# Join all
- Spark join `on` syntax:
    - on = ['column_name']
    - on = [col('table_alias_A.column_name') == col('table_alias_B.column_name')]
    - on = df_A.column_name == df_B.column_name
    - on = df_A['column_name'] == df_B['column_name']
- Spark join `how` syntax:
    - 'inner' is the default condition

In [111]:
spark.conf.set("spark.sql.sources.bucketing.enabled", "true")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")

In [93]:

# Perform the joins sequentially with consistent syntax
joined_df = (spark.table("bootcamp.matches_bucketed3").alias("mb")
             .join(spark.table("bootcamp.match_details_bucketed3").alias("mdb"),
                   col("mb.match_id") == col("mdb.match_id"), "inner")
             .join(spark.table("bootcamp.medal_matches_players_bucketed3").alias("mmpb"),
                   (col("mdb.match_id") == col("mmpb.match_id")) & (col("mdb.player_gamertag") == col("mmpb.player_gamertag_mmp")) )
             .join(broadcast(medals), ["medal_id"], "inner")
             .join(broadcast(maps), col("mb.mapid") == maps["mapid"])
            )
joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [mapid#4968], [mapid#121], Inner, BuildRight, false
   :- Project [medal_id#5015L, match_id#4967, mapid#4968, is_team_game#4969, playlist_id#4970, completion_date#4971, match_id#4977, player_gamertag#4978, player_total_kills#4979, player_total_deaths#4980, match_id#5013, player_gamertag_mmp#5014, count#5016, sprite_uri#81, sprite_left#82, sprite_top#83, sprite_sheet_width#84, sprite_sheet_height#85, sprite_width#86, sprite_height#87, classification#88, description#89, name#90, difficulty#91]
   :  +- BroadcastHashJoin [medal_id#5015L], [medal_id#80L], Inner, BuildRight, false
   :     :- SortMergeJoin [match_id#4977, player_gamertag#4978], [match_id#5013, player_gamertag_mmp#5014], Inner
   :     :  :- Sort [match_id#4977 ASC NULLS FIRST, player_gamertag#4978 ASC NULLS FIRST], false, 0
   :     :  :  +- Exchange hashpartitioning(match_id#4977, player_gamertag#4978, 200), ENSURE_REQUIREMENTS, [plan_id=2765]
   

In [95]:
from pyspark.sql.functions import broadcast

# First create all the DataFrames with aliases separately

mdb = spark.table("bootcamp.match_details_bucketed3")
mmpb = spark.table("bootcamp.medal_matches_players_bucketed3")

# Perform the joins sequentially with consistent syntax
joined_df = spark.table("bootcamp.matches_bucketed3").alias('mb').join(
    mdb,
    col("mb.match_id") == mdb["match_id"],
    "inner"
).join(
    mmpb,
    (mdb["match_id"] == mmpb["match_id"]) & (mdb["player_gamertag"] == mmpb["player_gamertag_mmp"]),
    "inner"
).join(
    broadcast(medals),
    mmpb["medal_id"] == medals["medal_id"],  # Assuming medals DF has medal_id column
    "inner"
).join(
    broadcast(maps),
    col("mb.mapid") == maps["mapid"],  # Assuming maps DF has mapid column
    "inner"
)

joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [mapid#5295], [mapid#121], Inner, BuildRight, false
   :- BroadcastHashJoin [medal_id#5288L], [medal_id#80L], Inner, BuildRight, false
   :  :- SortMergeJoin [match_id#5278, player_gamertag#5279], [match_id#5286, player_gamertag_mmp#5287], Inner
   :  :  :- Sort [match_id#5278 ASC NULLS FIRST, player_gamertag#5279 ASC NULLS FIRST], false, 0
   :  :  :  +- Exchange hashpartitioning(match_id#5278, player_gamertag#5279, 200), ENSURE_REQUIREMENTS, [plan_id=2845]
   :  :  :     +- SortMergeJoin [match_id#5294], [match_id#5278], Inner
   :  :  :        :- Sort [match_id#5294 ASC NULLS FIRST], false, 0
   :  :  :        :  +- Exchange hashpartitioning(match_id#5294, 200), ENSURE_REQUIREMENTS, [plan_id=2837]
   :  :  :        :     +- Filter isnotnull(mapid#5295)
   :  :  :        :        +- BatchScan demo.bootcamp.matches_bucketed3[match_id#5294, mapid#5295, is_team_game#5296, playlist_id#5297, completion_date#5298]

In [101]:

joined_df = (
    spark.table("bootcamp.matches_bucketed3").alias("mb")
    .join(spark.table("bootcamp.match_details_bucketed3").alias("mdb"), on = col("mb.match_id") == col("mdb.match_id"), how = "inner")
    .join(spark.table("bootcamp.medal_matches_players_bucketed3").alias("mmpb"), 
          on = (col("mdb.match_id") == col("mmpb.match_id")) & (col("mdb.player_gamertag") == col("mmpb.player_gamertag_mmp")), how = "inner")
    .join(broadcast(medals), on = ["medal_id"], how = "inner")
    .join(broadcast(maps), on = ["mapid"], how = "inner")
)

joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [mapid#5810, medal_id#5857L, match_id#5809, is_team_game#5811, playlist_id#5812, completion_date#5813, match_id#5819, player_gamertag#5820, player_total_kills#5821, player_total_deaths#5822, match_id#5855, player_gamertag_mmp#5856, count#5858, sprite_uri#81, sprite_left#82, sprite_top#83, sprite_sheet_width#84, sprite_sheet_height#85, sprite_width#86, sprite_height#87, classification#88, description#89, name#90, difficulty#91, ... 2 more fields]
   +- BroadcastHashJoin [mapid#5810], [mapid#121], Inner, BuildRight, false
      :- Project [medal_id#5857L, match_id#5809, mapid#5810, is_team_game#5811, playlist_id#5812, completion_date#5813, match_id#5819, player_gamertag#5820, player_total_kills#5821, player_total_deaths#5822, match_id#5855, player_gamertag_mmp#5856, count#5858, sprite_uri#81, sprite_left#82, sprite_top#83, sprite_sheet_width#84, sprite_sheet_height#85, sprite_width#86, sprite_height#87, classification#88,

In [98]:
joined_df.printSchema()

root
 |-- mapid: string (nullable = true)
 |-- medal_id: long (nullable = true)
 |-- match_id: string (nullable = true)
 |-- is_team_game: boolean (nullable = true)
 |-- playlist_id: string (nullable = true)
 |-- completion_date: timestamp (nullable = true)
 |-- match_id: string (nullable = true)
 |-- player_gamertag: string (nullable = true)
 |-- player_total_kills: integer (nullable = true)
 |-- player_total_deaths: integer (nullable = true)
 |-- match_id: string (nullable = true)
 |-- player_gamertag_mmp: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- sprite_uri: string (nullable = true)
 |-- sprite_left: integer (nullable = true)
 |-- sprite_top: integer (nullable = true)
 |-- sprite_sheet_width: integer (nullable = true)
 |-- sprite_sheet_height: integer (nullable = true)
 |-- sprite_width: integer (nullable = true)
 |-- sprite_height: integer (nullable = true)
 |-- classification: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: 

In [102]:
joined_df.count() # 6885858 join only match id ; 751134 join by both match id and player_gamertag

751134

In [105]:
spark.sql("""DROP TABLE IF EXISTS bootcamp.medal_matches_players_maps""")
bucketedDetailsDDL = """
CREATE OR REPLACE TABLE bootcamp.medal_matches_players_maps (
    mapid string,
    match_id STRING,
    medal_id BIGINT,
    is_team_game boolean,
    playlist_id string,
    completion_date timestamp,
    player_gamertag string,
    player_total_kills integer,
    player_total_deaths integer,
    count INTEGER,
    sprite_uri string,
    sprite_left integer,
    sprite_top integer,
    sprite_sheet_width integer,
    sprite_sheet_height integer,
    sprite_width integer,
    sprite_height integer,
    classification string,
    description string,
    name string,
    difficulty integer,
    map_name string,
    map_description string
)
USING iceberg
PARTITIONED BY (mapid);
"""

spark.sql(bucketedDetailsDDL)

DataFrame[]

In [107]:
joined_df.select('mapid', 'mb.match_id', 'medal_id', 'is_team_game', 'playlist_id', 'completion_date', 'player_gamertag', 
                 'player_total_kills', 'player_total_deaths','count','sprite_uri','sprite_left','sprite_top',
                 'sprite_sheet_width', 'sprite_sheet_height','sprite_width','sprite_height','classification',
                 'description', 'name', 'difficulty', 'map_name', 'map_description'
                )

DataFrame[mapid: string, match_id: string, medal_id: bigint, is_team_game: boolean, playlist_id: string, completion_date: timestamp, player_gamertag: string, player_total_kills: int, player_total_deaths: int, count: int, sprite_uri: string, sprite_left: int, sprite_top: int, sprite_sheet_width: int, sprite_sheet_height: int, sprite_width: int, sprite_height: int, classification: string, description: string, name: string, difficulty: int, map_name: string, map_description: string]

In [108]:
(
joined_df
    .select('mapid', 'mb.match_id', 'medal_id', 'is_team_game', 'playlist_id', 'completion_date', 'player_gamertag', 
                 'player_total_kills', 'player_total_deaths','count','sprite_uri','sprite_left','sprite_top',
                 'sprite_sheet_width', 'sprite_sheet_height','sprite_width','sprite_height','classification',
                 'description', 'name', 'difficulty', 'map_name', 'map_description'
    )
    .write.mode('append').partitionBy('mapid').saveAsTable('bootcamp.medal_matches_players_maps')
)

In [114]:
spark.table('bootcamp.medal_matches_players_maps').count() # 751134

751134

# Aggregations
  - Aggregate the joined data frame to figure out questions like:
    - Which player averages the most kills per game?
    - Which playlist gets played the most?
    - Which map gets played the most?
    - Which map do players get the most Killing Spree medals on?
  - With the aggregated data set
    - Try different `.sortWithinPartitions` to see which has the smallest data size (hint: playlists and maps are both very low cardinality)

In [115]:
from pyspark.sql.functions import *

df_mmpm = spark.table('bootcamp.medal_matches_players_maps')

In [116]:
df_mmpm.show(5)

+--------------------+--------------------+----------+------------+--------------------+-------------------+---------------+------------------+-------------------+-----+--------------------+-----------+----------+------------------+-------------------+------------+-------------+-----------------+--------------------+------------+----------+--------+---------------+
|               mapid|            match_id|  medal_id|is_team_game|         playlist_id|    completion_date|player_gamertag|player_total_kills|player_total_deaths|count|          sprite_uri|sprite_left|sprite_top|sprite_sheet_width|sprite_sheet_height|sprite_width|sprite_height|   classification|         description|        name|difficulty|map_name|map_description|
+--------------------+--------------------+----------+------------+--------------------+-------------------+---------------+------------------+-------------------+-----+--------------------+-----------+----------+------------------+-------------------+------------

## Which player averages the most kills per game?
- Answer: gimpinator14 - 109 avg kills

In [119]:
(df_mmpm
 .groupBy('player_gamertag', 'match_id')
 .agg(avg('player_total_kills').alias('avg_total_kills_game'))
 .sort(desc('avg_total_kills_game'))
 .limit(10)
 .show()
)

[Stage 124:==========================================>              (3 + 1) / 4]

+---------------+--------------------+--------------------+
|player_gamertag|            match_id|avg_total_kills_game|
+---------------+--------------------+--------------------+
|   gimpinator14|acf0e47e-20ac-4b1...|               109.0|
|  I Johann117 I|3ec9c9c9-4924-45e...|                96.0|
|   TameablePoet|39dfd6f5-f4ef-499...|                94.0|
|DanZ R3van DowN|fc6641b9-963e-43c...|                90.0|
|BudgetLegendary|5dcac8ab-425a-487...|                83.0|
|  DBossCnDTEXAS|73348a90-2121-42b...|                82.0|
|     Profit TKO|1ced37ad-f974-41c...|                76.0|
|      GsFurreal|acf0e47e-20ac-4b1...|                75.0|
|   Sexy is Back|3340d440-a2c4-42d...|                73.0|
|  Vice VersALEX|a870be98-b029-489...|                71.0|
+---------------+--------------------+--------------------+



In [120]:
(df_mmpm
 .groupBy('player_gamertag', 'match_id')
 .agg(avg('player_total_kills').alias('avg_total_kills_game'))
 .sortWithinPartitions(desc('avg_total_kills_game'))
 .limit(10)
 .show()
)

[Stage 127:==========================================>              (3 + 1) / 4]

+---------------+--------------------+--------------------+
|player_gamertag|            match_id|avg_total_kills_game|
+---------------+--------------------+--------------------+
|DanZ R3van DowN|fc6641b9-963e-43c...|                90.0|
|      GsFurreal|acf0e47e-20ac-4b1...|                75.0|
|  Vice VersALEX|a870be98-b029-489...|                71.0|
|    HisLattice1|3ec9c9c9-4924-45e...|                66.0|
|     taurenmonk|acf0e47e-20ac-4b1...|                64.0|
|  Slippery Smif|83d345a3-d51a-459...|                64.0|
|WhiteMountainDC|acf0e47e-20ac-4b1...|                63.0|
|     MONKEYBAKE|0dc3eb0a-48a8-4e9...|                62.0|
|LEGENDARY link0|d218a5b9-e918-4ee...|                60.0|
|    ohh Replxys|2de708d2-bf2d-4cb...|                60.0|
+---------------+--------------------+--------------------+



# Which playlist gets played the most?
- Playlist played the most: f72e0ef0-7c4a-4307-af78-8e38dac3fdba

In [129]:
df_playlist = df_mmpm.groupBy('playlist_id').agg(count('match_id').alias('cnt_playlist_id')).sort(desc('cnt_playlist_id')).limit(3)

print(df_playlist.collect()[0])

df_playlist.show()


Row(playlist_id='f72e0ef0-7c4a-4307-af78-8e38dac3fdba', cnt_playlist_id=202489)
+--------------------+---------------+
|         playlist_id|cnt_playlist_id|
+--------------------+---------------+
|f72e0ef0-7c4a-430...|         202489|
|c98949ae-60a8-43d...|         107422|
|2323b76a-db98-4e0...|          92148|
+--------------------+---------------+



In [131]:
df_playlist = df_mmpm.groupBy('playlist_id').agg(count('match_id').alias('cnt_playlist_id')).sortWithinPartitions(desc('cnt_playlist_id')).limit(3)

print(df_playlist.collect()[0])

df_playlist.show()


Row(playlist_id='f72e0ef0-7c4a-4307-af78-8e38dac3fdba', cnt_playlist_id=202489)
+--------------------+---------------+
|         playlist_id|cnt_playlist_id|
+--------------------+---------------+
|f72e0ef0-7c4a-430...|         202489|
|c98949ae-60a8-43d...|         107422|
|2323b76a-db98-4e0...|          92148|
+--------------------+---------------+



# Which map gets played the most?
- Map played the most: Breakout Arena

In [132]:
df_mostmap = df_mmpm.groupBy('map_name').agg(count('match_id').alias('cnt_map_name')).sort(desc('cnt_map_name')).limit(3)

print(df_mostmap.collect()[0])

df_mostmap.show()


Row(map_name='Breakout Arena', cnt_map_name=186118)
+--------------+------------+
|      map_name|cnt_map_name|
+--------------+------------+
|Breakout Arena|      186118|
|        Alpine|      105658|
|       Glacier|       70182|
+--------------+------------+



In [133]:
df_mostmap = df_mmpm.groupBy('map_name').agg(count('match_id').alias('cnt_map_name')).sortWithinPartitions(desc('cnt_map_name')).limit(3)

print(df_mostmap.collect()[0])

df_mostmap.show()


Row(map_name='Breakout Arena', cnt_map_name=186118)
+--------------+------------+
|      map_name|cnt_map_name|
+--------------+------------+
|Breakout Arena|      186118|
|        Alpine|      105658|
|       Glacier|       70182|
+--------------+------------+



# Which map do players get the most Killing Spree medals on?
- map_name='Breakout Arena', cnt_map_name=6553

In [138]:
df_mmpm.filter("name = 'Killing Spree'").show(2)

+--------------------+--------------------+----------+------------+--------------------+-------------------+---------------+------------------+-------------------+-----+--------------------+-----------+----------+------------------+-------------------+------------+-------------+--------------+--------------------+-------------+----------+--------+---------------+
|               mapid|            match_id|  medal_id|is_team_game|         playlist_id|    completion_date|player_gamertag|player_total_kills|player_total_deaths|count|          sprite_uri|sprite_left|sprite_top|sprite_sheet_width|sprite_sheet_height|sprite_width|sprite_height|classification|         description|         name|difficulty|map_name|map_description|
+--------------------+--------------------+----------+------------+--------------------+-------------------+---------------+------------------+-------------------+-----+--------------------+-----------+----------+------------------+-------------------+------------+---

In [139]:
df_mostmap_ks = (df_mmpm
                 .filter("name = 'Killing Spree'")
                 .groupBy('map_name').agg(count('match_id').alias('cnt_map_name'))
                 .sort(desc('cnt_map_name')).limit(3)
                )

print(df_mostmap_ks.collect()[0])

df_mostmap_ks.show()


Row(map_name='Breakout Arena', cnt_map_name=6553)
+--------------+------------+
|      map_name|cnt_map_name|
+--------------+------------+
|Breakout Arena|        6553|
|        Alpine|        4317|
|       Glacier|        2611|
+--------------+------------+



In [140]:
df_mostmap_ks = (df_mmpm
                 .filter("name = 'Killing Spree'")
                 .groupBy('map_name').agg(count('match_id').alias('cnt_map_name'))
                 .sortWithinPartitions(desc('cnt_map_name')).limit(3)
                )

print(df_mostmap_ks.collect()[0])

df_mostmap_ks.show()


Row(map_name='Breakout Arena', cnt_map_name=6553)
+--------------+------------+
|      map_name|cnt_map_name|
+--------------+------------+
|Breakout Arena|        6553|
|        Alpine|        4317|
|       Glacier|        2611|
+--------------+------------+

